# Data Augmenting

In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [3]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.Conv1 = nn.Conv2d(3,32, kernel_size=3, padding=1)
        self.Conv2 = nn.Conv2d(32,64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64*8*8, 128)
        self.fc2 = nn.Linear(128,10)

        self.pool = nn.MaxPool2d(2,2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self,x):
        x = self.pool(self.relu(self.Conv1(x)))
        x = self.pool(self.relu(self.Conv2(x)))
        x = x.view(-1,64*8*8)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [4]:
def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(train_loader)

In [5]:
def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad() :
        for images, labels in test_loader:
            images, labels = images.to(device),labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels. size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct/total

In [6]:
def load_data(data_augmentation=False):
    transforms_original = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    transforms_augmented = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    if data_augmentation:
        original_dataset = datasets.CIFAR10(root='/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data', train= True, download=True, transform=transforms_original)
        augmented_dataset = datasets.CIFAR10(root='/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data', train= True, download=True, transform=transforms_augmented)
        train_dataset = torch.utils.data.ConcatDataset([original_dataset, augmented_dataset])
    else:
        train_dataset = datasets.CIFAR10(root='/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data', train= True, download=True, transform=transforms_original)

    test_dataset = datasets.CIFAR10(root='/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data', train= False, download=True, transform=transforms_original)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=3)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=3)

    return train_loader, test_loader

In [7]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
epochs= 25

In [11]:
train_loader, test_loader = load_data(data_augmentation=False)

In [19]:
from tqdm import tqdm

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Without Augmentation")
print('-'*100)

for epoch in range(epochs):
    train_loader_tqdm = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    loss = train_model(model, train_loader_tqdm, criterion, optimizer, device)
    
    print(f'Epoch: {epoch+1} Loss: {loss}')

accuracy = evaluate_model(model, test_loader, device)
print(f'Accuracy:{accuracy}%')

Without Augmentation
----------------------------------------------------------------------------------------------------


Epoch 1/25:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 1/25: 100%|██████████| 782/782 [00:27<00:00, 27.94it/s] 


Epoch: 1 Loss: 1.5756470745481799


Epoch 2/25: 100%|██████████| 782/782 [00:26<00:00, 29.93it/s] 


...

Epoch 24/25: 100%|██████████| 782/782 [00:26<00:00, 29.51it/s] 


Epoch: 24 Loss: 0.45895331625438407


Epoch 25/25: 100%|██████████| 782/782 [00:28<00:00, 27.11it/s] 

Epoch: 25 Loss: 0.45684439424053785


Accuracy:72.33%


In [8]:
train_loader, test_loader = load_data(data_augmentation=True) 

In [11]:
from tqdm import tqdm

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("With Augmentation")
print('-'*100)

for epoch in range(epochs):
    train_loader_tqdm = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    loss = train_model(model, train_loader, criterion, optimizer, device)
    print(f'Epoch: {epoch+1} Loss: {loss}')

accuracy = evaluate_model(model, test_loader, device)
print(f'Accuracy:{accuracy}%')

With Augmentation
----------------------------------------------------------------------------------------------------


Epoch 1/25:   0%|          | 0/1563 [00:00<?, ?it/s]

Epoch: 1 Loss: 1.5293882059425554


Epoch 1/25:   0%|          | 0/1563 [00:37<?, ?it/s]


Epoch: 2 Loss: 1.2415470460326108


...

Epoch: 23 Loss: 0.7794995166442368


Epoch 23/25:   0%|          | 0/1563 [00:33<?, ?it/s]


Epoch: 24 Loss: 0.776067505585255


Epoch 24/25:   0%|          | 0/1563 [00:35<?, ?it/s]


Epoch: 25 Loss: 0.7687946899304845
Accuracy:76.89%
